# Hybrid WaveNet-TimeGAN for Synthetic Financial Data

This notebook implements a **hybrid architecture** combining:
- **WaveNet blocks** (dilated causal convolutions with gated activations) as the core temporal unit
- **TimeGAN framework** (embedder/recovery/supervisor/generator/discriminator) with 3-phase training

### Why Hybrid?
| Component | Original TimeGAN | LSTM TimeGAN (nb 03) | Pure WaveNet GAN (nb 07) | **This Hybrid** |
|-----------|-----------------|---------------------|-------------------------|----------------|
| Core unit | GRU | LSTM | Dilated causal conv | **Dilated causal conv** |
| Embedder | ✅ | ✅ | ❌ | **✅** |
| Recovery | ✅ | ✅ | ❌ | **✅** |
| Supervisor | ✅ | ❌ | ❌ | **✅** |
| Training phases | 3 | 2 | 1 | **3** |
| Receptive field | Sequential | Sequential | Exponential (2^n) | **Exponential (2^n)** |
| Skip connections | ❌ | ❌ | ✅ | **✅** |
| Gated activations | ❌ | ❌ | tanh⊙sigmoid | **tanh⊙sigmoid** |

### Architecture
```
Phase 1 — Embedding:   X → [WaveNet Embedder] → H → [WaveNet Recovery] → X̂
Phase 2 — Supervised:  H_t → [WaveNet Supervisor] → Ĥ_{t+1}  (temporal coherence)
Phase 3 — Joint:       Z → [WaveNet Generator] → Ê → [Recovery] → X̃
                        + Discriminator judges in latent space H vs Ê
                        + Supervisor loss for temporal consistency
```

## 1. Setup and Data Loading

In [1]:
import os
import site
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

# Suppress TF C++ info/warning spam (must be set BEFORE importing TF)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# XLA fix for pip-bundled CUDA toolkit
_cuda_nvcc_dir = os.path.join(
    site.getsitepackages()[0] if site.getsitepackages() else
    os.path.join(os.path.dirname(os.__file__), '..', 'site-packages'),
    'nvidia', 'cuda_nvcc'
)
os.environ['XLA_FLAGS'] = f'--xla_gpu_cuda_data_dir={_cuda_nvcc_dir}'

import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, Multiply, Add, Dense,
    GaussianNoise, LayerNormalization, Activation,
    Dropout                                          # ← added for discriminator
)
from tensorflow.keras.models import Model

tf.random.set_seed(42)
np.random.seed(42)

# Also suppress Python-level TF logs
tf.get_logger().setLevel('ERROR')

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# Load SP500 data — same preprocessing as notebooks 03 and 07
data = pd.read_csv('../data/raw/sp500.csv', index_col='Date', parse_dates=True)
data = data.apply(pd.to_numeric, errors='coerce')

close_prices = data['Close']
log_returns = np.log(close_prices / close_prices.shift(1)).dropna()
log_returns_array = log_returns.values.reshape(-1, 1)

scaler = MinMaxScaler()
log_returns_scaled = scaler.fit_transform(log_returns_array)

sequence_length = 24
sequences = []
for i in range(len(log_returns_scaled) - sequence_length):
    sequences.append(log_returns_scaled[i:i + sequence_length])

sequences = np.array(sequences, dtype=np.float32)
print(f"Data shape: {sequences.shape}  (samples, timesteps, features)")
print(f"Log returns — min: {log_returns.min():.4f}, max: {log_returns.max():.4f}")

Data shape: (6012, 24, 1)  (samples, timesteps, features)
Log returns — min: -0.1277, max: 0.1096


## 2. WaveNet Building Blocks

Same dilated causal convolution blocks as notebook 07, used inside all 5 TimeGAN networks.

In [3]:
###############################################
# WaveNet Residual Block with Optional Gaussian Noise
# Adapted from: wavenet_gan_uncond_noise.py
###############################################
def wavenet_residual_block(input_tensor, nfilt, dilation_rate, 
                           residual_noise_std=None, seed=None):
    """Gated causal convolution block with skip connection.
    
    The core building block: uses tanh * sigmoid gating (like the 
    original WaveNet paper) instead of plain activations.
    """
    x = input_tensor
    
    # Project to nfilt channels if needed
    if x.shape[-1] != nfilt:
        x = Conv1D(filters=nfilt, kernel_size=1, padding='same')(x)
    
    # Optional noise injection for regularization
    if residual_noise_std:
        x = GaussianNoise(stddev=residual_noise_std, seed=seed)(x)
    
    # Gated activation: tanh(conv) * sigmoid(conv)
    tanh_out = Conv1D(filters=nfilt, kernel_size=3, dilation_rate=dilation_rate,
                      padding='causal', activation='tanh')(x)
    sigm_out = Conv1D(filters=nfilt, kernel_size=3, dilation_rate=dilation_rate,
                      padding='causal', activation='sigmoid')(x)
    gated = Multiply()([tanh_out, sigm_out])
    
    # Skip and residual outputs
    skip_out = Conv1D(filters=nfilt, kernel_size=1, padding='same')(gated)
    residual = Conv1D(filters=nfilt, kernel_size=1, padding='same')(gated)
    residual_out = Add()([x, residual])
    
    return residual_out, skip_out


###############################################
# WaveNet Block — stack of dilated residual blocks
###############################################
def wavenet_block(input_tensor, nfilt, dilation_rates=None,
                  residual_noise_std=None, seed=None):
    """Stack of dilated causal convolutions with exponentially increasing dilation.
    
    Receptive field = sum(dilation_rates) * (kernel_size - 1) + 1
    For [1,2,4,8,16]: receptive field = 31 * 2 + 1 = 63 (covers seq_len=24)
    """
    if dilation_rates is None:
        dilation_rates = [1, 2, 4, 8, 16]  # Reduced for seq_len=24
    
    skip_connections = []
    x = input_tensor
    for i, dilation in enumerate(dilation_rates):
        x, skip = wavenet_residual_block(
            x, nfilt, dilation,
            residual_noise_std=residual_noise_std,
            seed=(seed + i) if seed else None
        )
        skip_connections.append(skip)
    return Add()(skip_connections)


###############################################
# Deep WaveNet — multiple WaveNet blocks stacked
###############################################
def deep_wavenet(input_tensor, nfilt, n_stacks=2,
                 residual_noise_std=None, seed=None):
    """Stack multiple WaveNet blocks for deeper temporal modeling."""
    x = input_tensor
    for i in range(n_stacks):
        x = wavenet_block(
            x, nfilt,
            residual_noise_std=residual_noise_std,
            seed=(seed + 100 + i) if seed else None
        )
    return x


print("WaveNet building blocks defined.")

WaveNet building blocks defined.


## 3. Build All 5 TimeGAN Networks (WaveNet-based)

Following the original TimeGAN paper (Yoon et al., 2019):
1. **Embedder** — maps data space → latent space
2. **Recovery** — maps latent space → data space (decoder)
3. **Supervisor** — models temporal dynamics in latent space ($h_t \to h_{t+1}$)
4. **Generator** — produces synthetic latent sequences from noise
5. **Discriminator** — distinguishes real vs fake in latent space

In [4]:
# ============================================================
# Hyperparameters
# ============================================================
FEATURE_DIM = 1        # Single feature: log return
SEQUENCE_LENGTH = 24   # Match all other notebooks
HIDDEN_DIM = 24        # Latent/embedding dimension
NFILT = 32             # WaveNet filter width
N_STACKS = 2           # WaveNet stacking depth
LATENT_DIM = 24        # Generator noise dim (match HIDDEN_DIM for supervisor compatibility)

# --- Learning rates — match NB07 (were 2.5× too high) ---
LR_GENERATOR = 0.0002       # was 0.0005
LR_DISCRIMINATOR = 0.0001   # was 0.00025 (half of G LR, same as NB07)
BETA_1 = 0.5
BATCH_SIZE = 128

# --- Regularisation — match NB07 ---
RESIDUAL_NOISE_STD = 0.01   # GaussianNoise in every WaveNet residual block
LATENT_NOISE_STD = 0.01     # GaussianNoise on generator input
DROPOUT_RATE = 0.1           # Dropout in discriminator before output

# Training epochs per phase (with early stopping patience)
EMBEDDING_EPOCHS = 100
SUPERVISED_EPOCHS = 150
JOINT_EPOCHS = 1000
EARLY_STOP_PATIENCE = 20  # Stop phase 1/2 if no improvement for N epochs

# Loss weights for joint phase — match NB02 scaling (100×)
LAMBDA_SUPERVISED = 100.0       # Supervisor loss weight (applied to sqrt)
LAMBDA_RECONSTRUCTION = 100.0   # Moment matching loss weight
D_SKIP_THRESHOLD = 1.0          # Skip D when d_loss < threshold

print(f"Total training: {EMBEDDING_EPOCHS + SUPERVISED_EPOCHS + JOINT_EPOCHS} epochs "
      f"({EMBEDDING_EPOCHS} embed + {SUPERVISED_EPOCHS} supervised + {JOINT_EPOCHS} joint)")
print(f"LR_G: {LR_GENERATOR}, LR_D: {LR_DISCRIMINATOR}")
print(f"Regularisation: residual_noise={RESIDUAL_NOISE_STD}, "
      f"latent_noise={LATENT_NOISE_STD}, dropout={DROPOUT_RATE}")
print(f"Lambda supervised: {LAMBDA_SUPERVISED}, Lambda reconstruction: {LAMBDA_RECONSTRUCTION}")
print(f"D-skip threshold: {D_SKIP_THRESHOLD}, Early stop patience: {EARLY_STOP_PATIENCE}")

Total training: 1250 epochs (100 embed + 150 supervised + 1000 joint)
LR_G: 0.0002, LR_D: 0.0001
Regularisation: residual_noise=0.01, latent_noise=0.01, dropout=0.1
Lambda supervised: 100.0, Lambda reconstruction: 100.0
D-skip threshold: 1.0, Early stop patience: 20


In [5]:
# ============================================================
# 1. EMBEDDER: X (data space) → H (latent space)
# ============================================================
def build_embedder(seq_len, feature_dim, hidden_dim, nfilt, n_stacks,
                   residual_noise_std=None, seed=None):
    inp = Input(shape=(seq_len, feature_dim), name='embedder_input')
    x = deep_wavenet(inp, nfilt, n_stacks,
                     residual_noise_std=residual_noise_std, seed=seed)
    out = Dense(hidden_dim, activation='sigmoid', name='embedder_output')(x)
    return Model(inp, out, name='WaveNet_Embedder')


# ============================================================
# 2. RECOVERY: H (latent space) → X̂ (data space)
# ============================================================
def build_recovery(seq_len, hidden_dim, feature_dim, nfilt, n_stacks,
                   residual_noise_std=None, seed=None):
    inp = Input(shape=(seq_len, hidden_dim), name='recovery_input')
    x = deep_wavenet(inp, nfilt, n_stacks,
                     residual_noise_std=residual_noise_std, seed=seed)
    out = Dense(feature_dim, activation='sigmoid', name='recovery_output')(x)
    return Model(inp, out, name='WaveNet_Recovery')


# ============================================================
# 3. SUPERVISOR: H_t → Ĥ_{t+1} (temporal dynamics in latent space)
# Uses a causal WaveNet so output at time t only sees t and before,
# naturally predicting the "next" representation.
# ============================================================
def build_supervisor(seq_len, hidden_dim, nfilt, n_stacks,
                     residual_noise_std=None, seed=None):
    inp = Input(shape=(seq_len, hidden_dim), name='supervisor_input')
    x = deep_wavenet(inp, nfilt, n_stacks,
                     residual_noise_std=residual_noise_std, seed=seed)
    out = Dense(hidden_dim, activation='sigmoid', name='supervisor_output')(x)
    return Model(inp, out, name='WaveNet_Supervisor')


# ============================================================
# 4. GENERATOR: Z (noise) → Ê (fake latent sequences)
# Generates in latent space, NOT data space.
# GaussianNoise on input matches NB07 regularisation.
# ============================================================
def build_generator(seq_len, latent_dim, hidden_dim, nfilt, n_stacks,
                    latent_noise_std=None, residual_noise_std=None, seed=None):
    inp = Input(shape=(seq_len, latent_dim), name='generator_noise_input')
    x = inp
    if latent_noise_std:
        x = GaussianNoise(latent_noise_std, name='gen_latent_noise')(x)
    x = deep_wavenet(x, nfilt, n_stacks,
                     residual_noise_std=residual_noise_std, seed=seed)
    out = Dense(hidden_dim, activation='sigmoid', name='generator_output')(x)
    return Model(inp, out, name='WaveNet_Generator')


# ============================================================
# 5. DISCRIMINATOR: H or Ê → real/fake score (in latent space)
# Dropout after WaveNet matches NB07 regularisation.
# ============================================================
def build_discriminator(seq_len, hidden_dim, nfilt, n_stacks,
                        dropout_rate=None, residual_noise_std=None, seed=None):
    inp = Input(shape=(seq_len, hidden_dim), name='discriminator_input')
    x = deep_wavenet(inp, nfilt, n_stacks,
                     residual_noise_std=residual_noise_std, seed=seed)
    if dropout_rate:
        x = Dropout(dropout_rate, seed=seed, name='disc_dropout')(x)
    out = Dense(1, name='real_fake_logit')(x)  # No activation — use from_logits=True
    return Model(inp, out, name='WaveNet_Discriminator')


print("All 5 network builders defined (with regularisation params).")

All 5 network builders defined (with regularisation params).


In [6]:
# ============================================================
# Instantiate models  (with NB07-matching regularisation)
# ============================================================
_noise_kw = dict(residual_noise_std=RESIDUAL_NOISE_STD, seed=42)

embedder      = build_embedder(SEQUENCE_LENGTH, FEATURE_DIM, HIDDEN_DIM,
                               NFILT, N_STACKS, **_noise_kw)
recovery      = build_recovery(SEQUENCE_LENGTH, HIDDEN_DIM, FEATURE_DIM,
                               NFILT, N_STACKS, **_noise_kw)
supervisor    = build_supervisor(SEQUENCE_LENGTH, HIDDEN_DIM,
                                 NFILT, N_STACKS, **_noise_kw)
generator     = build_generator(SEQUENCE_LENGTH, LATENT_DIM, HIDDEN_DIM,
                                NFILT, N_STACKS,
                                latent_noise_std=LATENT_NOISE_STD, **_noise_kw)
discriminator = build_discriminator(SEQUENCE_LENGTH, HIDDEN_DIM,
                                    NFILT, N_STACKS,
                                    dropout_rate=DROPOUT_RATE, **_noise_kw)

# Print parameter counts
for m in [embedder, recovery, supervisor, generator, discriminator]:
    print(f"{m.name:<25s} params: {m.count_params():>8,}")
total = sum(m.count_params() for m in [embedder, recovery, supervisor, generator, discriminator])
print(f"{'TOTAL':<25s} params: {total:>8,}")

I0000 00:00:1772751040.963299   69059 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2287 MB memory:  -> device: 0, name: Quadro P620, pci bus id: 0000:01:00.0, compute capability: 6.1


WaveNet_Embedder          params:   81,944
WaveNet_Recovery          params:   81,921
WaveNet_Supervisor        params:   82,680
WaveNet_Generator         params:   82,680
WaveNet_Discriminator     params:   81,921
TOTAL                     params:  411,146


In [7]:
import sys, datetime
sys.path.append('..')
from utils.models_utils import smooth_positive_labels, smooth_negative_labels

# ============================================================
# Optimizers — separate LRs matching NB07
# ============================================================
embedder_optimizer    = tf.keras.optimizers.Adam(learning_rate=LR_GENERATOR)
supervisor_optimizer  = tf.keras.optimizers.Adam(learning_rate=LR_GENERATOR)
generator_optimizer   = tf.keras.optimizers.Adam(learning_rate=LR_GENERATOR, beta_1=BETA_1)
discriminator_optimizer = tf.keras.optimizers.Adam(learning_rate=LR_DISCRIMINATOR, beta_1=BETA_1)

# Loss
mse = tf.keras.losses.MeanSquaredError()
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

# ============================================================
# TensorBoard — log all training phases to a single run directory
# ============================================================
log_dir = os.path.join('..', 'logs', 'wavenet_timegan_hybrid',
                       datetime.datetime.now().strftime('%Y%m%d-%H%M%S'))
tb_writer = tf.summary.create_file_writer(log_dir)

print(f"Optimizers, losses, and label smoothing configured.")
print(f"  LR_G = {LR_GENERATOR},  LR_D = {LR_DISCRIMINATOR}")
print(f"TensorBoard log dir: {log_dir}")
print(f"  → Launch: tensorboard --logdir {os.path.dirname(log_dir)}")

Optimizers, losses, and label smoothing configured.
  LR_G = 0.0002,  LR_D = 0.0001
TensorBoard log dir: ../logs/wavenet_timegan_hybrid/20260305-155045
  → Launch: tensorboard --logdir ../logs/wavenet_timegan_hybrid


## 4. Phase 1 — Embedding Training

Train embedder + recovery to learn a faithful latent representation.  
Loss: $\mathcal{L}_E = \|X - \text{Recovery}(\text{Embedder}(X))\|^2$

In [8]:
@tf.function
def train_embedding_step(real_data):
    """Phase 1: Train embedder + recovery via reconstruction loss."""
    with tf.GradientTape() as tape:
        # Forward: X → H → X̂
        H = embedder(real_data, training=True)
        X_hat = recovery(H, training=True)

        # Reconstruction loss
        e_loss_reconstruction = mse(real_data, X_hat)

        # Supervised loss on embeddings (temporal consistency)
        H_supervised = supervisor(H, training=True)
        e_loss_supervised = mse(H[:, 1:, :], H_supervised[:, :-1, :])

        e_loss = e_loss_reconstruction + 0.1 * e_loss_supervised

    # Update embedder, recovery, and supervisor
    e_vars = embedder.trainable_variables + recovery.trainable_variables + supervisor.trainable_variables
    grads = tape.gradient(e_loss, e_vars)
    grads = [tf.clip_by_value(g, -1.0, 1.0) if g is not None else g for g in grads]
    embedder_optimizer.apply_gradients(zip(grads, e_vars))

    return e_loss_reconstruction, e_loss_supervised

In [9]:
# Phase 1 training
dataset = tf.data.Dataset.from_tensor_slices(sequences)
dataset = dataset.shuffle(len(sequences), seed=42).batch(BATCH_SIZE, drop_remainder=True).prefetch(tf.data.AUTOTUNE)

print(f"Phase 1: Embedding Training ({EMBEDDING_EPOCHS} epochs, early stop patience={EARLY_STOP_PATIENCE})")
print("=" * 60)

embedding_history = {'reconstruction': [], 'supervised': []}
best_recon_loss = float('inf')
patience_counter = 0

for epoch in range(EMBEDDING_EPOCHS):
    epoch_recon, epoch_sup = [], []
    for batch in dataset:
        recon_loss, sup_loss = train_embedding_step(batch)
        epoch_recon.append(recon_loss.numpy())
        epoch_sup.append(sup_loss.numpy())

    avg_recon = np.mean(epoch_recon)
    avg_sup = np.mean(epoch_sup)
    embedding_history['reconstruction'].append(avg_recon)
    embedding_history['supervised'].append(avg_sup)

    # TensorBoard logging
    with tb_writer.as_default():
        tf.summary.scalar('phase1/reconstruction_loss', avg_recon, step=epoch)
        tf.summary.scalar('phase1/supervised_loss', avg_sup, step=epoch)
        tf.summary.scalar('phase1/total_loss', avg_recon + 0.1 * avg_sup, step=epoch)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:4d}/{EMBEDDING_EPOCHS} | "
              f"Recon: {avg_recon:.6f} | "
              f"Supervised: {avg_sup:.6f}")

    # Early stopping
    if avg_recon < best_recon_loss - 1e-6:
        best_recon_loss = avg_recon
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"  Early stopping at epoch {epoch} (no improvement for {EARLY_STOP_PATIENCE} epochs)")
            break

tb_writer.flush()
print(f"\nPhase 1 complete after {len(embedding_history['reconstruction'])} epochs. "
      f"Final recon loss: {embedding_history['reconstruction'][-1]:.6f}")

Phase 1: Embedding Training (100 epochs, early stop patience=20)
Epoch    0/100 | Recon: 0.003247 | Supervised: 0.001929
Epoch   10/100 | Recon: 0.000149 | Supervised: 0.000434
Epoch   20/100 | Recon: 0.000096 | Supervised: 0.000290
Epoch   30/100 | Recon: 0.000074 | Supervised: 0.000227
Epoch   40/100 | Recon: 0.000063 | Supervised: 0.000191
Epoch   50/100 | Recon: 0.000052 | Supervised: 0.000168
Epoch   60/100 | Recon: 0.000046 | Supervised: 0.000151
Epoch   70/100 | Recon: 0.000043 | Supervised: 0.000138
Epoch   80/100 | Recon: 0.000033 | Supervised: 0.000128
Epoch   90/100 | Recon: 0.000033 | Supervised: 0.000121

Phase 1 complete after 100 epochs. Final recon loss: 0.000030


## 5. Phase 2 — Supervised Training

Train supervisor to model temporal dynamics: $h_t \to \hat{h}_{t+1}$  
Loss: $\mathcal{L}_S = \|h_{t+1} - \text{Supervisor}(h_t)\|^2$

In [10]:
@tf.function
def train_supervisor_step(real_data):
    """Phase 2: Train supervisor on real embeddings."""
    with tf.GradientTape() as tape:
        # Embed real data (embedder frozen in terms of purpose,
        # but we include it to allow gentle co-adaptation)
        H = embedder(real_data, training=False)

        # Supervisor predicts next-step embeddings
        H_hat = supervisor(H, training=True)

        # Temporal loss: H[:, 1:, :] should match supervisor output at [:, :-1, :]
        s_loss = mse(H[:, 1:, :], H_hat[:, :-1, :])

    s_vars = supervisor.trainable_variables
    grads = tape.gradient(s_loss, s_vars)
    grads = [tf.clip_by_value(g, -1.0, 1.0) if g is not None else g for g in grads]
    supervisor_optimizer.apply_gradients(zip(grads, s_vars))

    return s_loss

In [11]:
print(f"Phase 2: Supervised Training ({SUPERVISED_EPOCHS} epochs, early stop patience={EARLY_STOP_PATIENCE})")
print("=" * 60)

supervised_history = []
best_sup_loss = float('inf')
patience_counter = 0

for epoch in range(SUPERVISED_EPOCHS):
    epoch_losses = []
    for batch in dataset:
        s_loss = train_supervisor_step(batch)
        epoch_losses.append(s_loss.numpy())

    avg_sup = np.mean(epoch_losses)
    supervised_history.append(avg_sup)

    # TensorBoard logging
    with tb_writer.as_default():
        tf.summary.scalar('phase2/supervisor_loss', avg_sup, step=epoch)

    if epoch % 10 == 0:
        print(f"Epoch {epoch:4d}/{SUPERVISED_EPOCHS} | "
              f"Supervisor loss: {avg_sup:.6f}")

    # Early stopping
    if avg_sup < best_sup_loss - 1e-6:
        best_sup_loss = avg_sup
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"  Early stopping at epoch {epoch} (no improvement for {EARLY_STOP_PATIENCE} epochs)")
            break

tb_writer.flush()
print(f"\nPhase 2 complete after {len(supervised_history)} epochs. "
      f"Final supervisor loss: {supervised_history[-1]:.6f}")

Phase 2: Supervised Training (150 epochs, early stop patience=20)
Epoch    0/150 | Supervisor loss: 0.000098
Epoch   10/150 | Supervisor loss: 0.000095
Epoch   20/150 | Supervisor loss: 0.000093
Epoch   30/150 | Supervisor loss: 0.000093
Epoch   40/150 | Supervisor loss: 0.000092
Epoch   50/150 | Supervisor loss: 0.000092
Epoch   60/150 | Supervisor loss: 0.000090
  Early stopping at epoch 66 (no improvement for 20 epochs)

Phase 2 complete after 67 epochs. Final supervisor loss: 0.000091


## 6. Phase 3 — Joint Adversarial Training

Train all networks together:
- **Generator** produces fake latent sequences from noise
- **Supervisor** guides temporal coherence of generated sequences
- **Discriminator** judges real (embedded) vs fake (generated) in latent space
- **Embedder + Recovery** continue fine-tuning

Generator loss: $\mathcal{L}_G = \mathcal{L}_{adversarial} + \lambda_S \cdot \mathcal{L}_{supervised} + \lambda_R \cdot \mathcal{L}_{moment}$

In [12]:
@tf.function
def train_joint_generator_step(real_data, noise):
    """Phase 3: Joint training — generator side."""
    with tf.GradientTape() as tape:
        # Real path
        H = embedder(real_data, training=True)
        H_supervised = supervisor(H, training=True)
        X_hat = recovery(H, training=True)

        # Fake path: noise → generator → supervisor → recovery
        E_hat = generator(noise, training=True)
        H_hat_supervised = supervisor(E_hat, training=True)
        X_tilde = recovery(E_hat, training=True)

        # Discriminator on fake
        Y_fake = discriminator(H_hat_supervised, training=False)
        Y_fake_e = discriminator(E_hat, training=False)

        # ---- Generator losses ----

        # 1. Adversarial loss (fool discriminator) — with label smoothing
        gen_real_labels = smooth_positive_labels(tf.ones_like(Y_fake))
        gen_real_labels_e = smooth_positive_labels(tf.ones_like(Y_fake_e))
        g_loss_adv = bce(gen_real_labels, Y_fake)
        g_loss_adv_e = bce(gen_real_labels_e, Y_fake_e)

        # 2. Supervised loss — sqrt keeps gradient alive as MSE→0 (NB02 strategy)
        g_loss_sup_raw = mse(H[:, 1:, :], H_supervised[:, :-1, :])
        g_loss_sup = tf.sqrt(g_loss_sup_raw + 1e-8)

        # 3. Moment matching loss (mean + variance of real vs fake embeddings)
        real_mean = tf.reduce_mean(H, axis=[0, 1])
        fake_mean = tf.reduce_mean(E_hat, axis=[0, 1])
        real_var = tf.math.reduce_variance(H, axis=[0, 1])
        fake_var = tf.math.reduce_variance(E_hat, axis=[0, 1])
        g_loss_moment = (tf.reduce_mean(tf.abs(real_mean - fake_mean)) +
                         tf.reduce_mean(tf.abs(tf.sqrt(real_var + 1e-6) -
                                               tf.sqrt(fake_var + 1e-6))))

        # 4. Reconstruction loss — sqrt (same as NB02: 10 * sqrt(embedding_loss))
        e_loss_recon_raw = mse(real_data, X_hat)
        e_loss_recon = 10.0 * tf.sqrt(e_loss_recon_raw + 1e-8)

        # Combined generator loss
        g_loss = (g_loss_adv + g_loss_adv_e +
                  LAMBDA_SUPERVISED * g_loss_sup +
                  LAMBDA_RECONSTRUCTION * g_loss_moment +
                  e_loss_recon)

    # Update generator, supervisor, embedder, recovery
    g_vars = (generator.trainable_variables +
              supervisor.trainable_variables +
              embedder.trainable_variables +
              recovery.trainable_variables)
    grads = tape.gradient(g_loss, g_vars)
    grads = [tf.clip_by_value(g, -1.0, 1.0) if g is not None else g for g in grads]
    generator_optimizer.apply_gradients(zip(grads, g_vars))

    return g_loss_adv, g_loss_sup, g_loss_moment, e_loss_recon


@tf.function
def get_discriminator_loss(real_data, noise):
    """Compute D loss without updating weights (for D-skip check)."""
    H = embedder(real_data, training=False)
    H_supervised = supervisor(H, training=False)
    E_hat = generator(noise, training=False)
    H_hat_supervised = supervisor(E_hat, training=False)

    Y_real = discriminator(H, training=False)
    Y_fake = discriminator(H_hat_supervised, training=False)
    Y_fake_e = discriminator(E_hat, training=False)

    real_labels = smooth_positive_labels(tf.ones_like(Y_real))
    fake_labels = smooth_negative_labels(tf.zeros_like(Y_fake))
    fake_labels_e = smooth_negative_labels(tf.zeros_like(Y_fake_e))

    d_loss = (bce(real_labels, Y_real) +
              bce(fake_labels, Y_fake) +
              bce(fake_labels_e, Y_fake_e))
    return d_loss


@tf.function
def train_joint_discriminator_step(real_data, noise):
    """Phase 3: Joint training — discriminator side."""
    with tf.GradientTape() as tape:
        # Real path
        H = embedder(real_data, training=False)
        H_supervised = supervisor(H, training=False)

        # Fake path
        E_hat = generator(noise, training=False)
        H_hat_supervised = supervisor(E_hat, training=False)

        # Discriminator on real and fake
        Y_real = discriminator(H, training=True)
        Y_fake = discriminator(H_hat_supervised, training=True)
        Y_fake_e = discriminator(E_hat, training=True)

        # Discriminator losses — with label smoothing
        real_labels = smooth_positive_labels(tf.ones_like(Y_real))
        fake_labels = smooth_negative_labels(tf.zeros_like(Y_fake))
        fake_labels_e = smooth_negative_labels(tf.zeros_like(Y_fake_e))

        d_loss_real = bce(real_labels, Y_real)
        d_loss_fake = bce(fake_labels, Y_fake)
        d_loss_fake_e = bce(fake_labels_e, Y_fake_e)

        d_loss = d_loss_real + d_loss_fake + d_loss_fake_e

    d_vars = discriminator.trainable_variables
    grads = tape.gradient(d_loss, d_vars)
    grads = [tf.clip_by_value(g, -1.0, 1.0) if g is not None else g for g in grads]
    discriminator_optimizer.apply_gradients(zip(grads, d_vars))

    return d_loss_real, d_loss_fake

In [ ]:
print(f"Phase 3: Joint Adversarial Training ({JOINT_EPOCHS} epochs)")
print(f"  D-skip threshold: {D_SKIP_THRESHOLD}")
print("=" * 70)

joint_history = {
    'g_adv': [], 'g_sup': [], 'g_moment': [], 'e_recon': [],
    'd_real': [], 'd_fake': [], 'd_skipped': []
}

for epoch in range(JOINT_EPOCHS):
    epoch_metrics = {k: [] for k in joint_history}
    d_skip_count = 0
    batch_count = 0

    for batch in dataset:
        batch_size = tf.shape(batch)[0]
        noise = tf.random.normal([batch_size, SEQUENCE_LENGTH, LATENT_DIM])

        # Train generator (+ supervisor, embedder, recovery) twice per disc step
        for _ in range(2):
            g_adv, g_sup, g_moment, e_recon = train_joint_generator_step(batch, noise)

        # D-skip: only train discriminator if it's not already winning
        d_loss_check = get_discriminator_loss(batch, noise)
        if d_loss_check > D_SKIP_THRESHOLD:
            d_real, d_fake = train_joint_discriminator_step(batch, noise)
        else:
            d_skip_count += 1
            # Still log the current D outputs (without training)
            d_real = d_loss_check * 0.5  # approximate split for logging
            d_fake = d_loss_check * 0.5
        batch_count += 1

        epoch_metrics['g_adv'].append(g_adv.numpy() if hasattr(g_adv, 'numpy') else float(g_adv))
        epoch_metrics['g_sup'].append(g_sup.numpy() if hasattr(g_sup, 'numpy') else float(g_sup))
        epoch_metrics['g_moment'].append(g_moment.numpy() if hasattr(g_moment, 'numpy') else float(g_moment))
        epoch_metrics['e_recon'].append(e_recon.numpy() if hasattr(e_recon, 'numpy') else float(e_recon))
        epoch_metrics['d_real'].append(d_real.numpy() if hasattr(d_real, 'numpy') else float(d_real))
        epoch_metrics['d_fake'].append(d_fake.numpy() if hasattr(d_fake, 'numpy') else float(d_fake))

    # Compute epoch averages
    avgs = {k: np.mean(epoch_metrics[k]) for k in epoch_metrics if k != 'd_skipped'}
    skip_pct = 100.0 * d_skip_count / max(batch_count, 1)
    for k in joint_history:
        if k != 'd_skipped':
            joint_history[k].append(avgs[k])
    joint_history['d_skipped'].append(skip_pct)

    # TensorBoard logging — all Phase 3 losses
    with tb_writer.as_default():
        tf.summary.scalar('phase3/g_adversarial', avgs['g_adv'], step=epoch)
        tf.summary.scalar('phase3/g_supervised', avgs['g_sup'], step=epoch)
        tf.summary.scalar('phase3/g_moment', avgs['g_moment'], step=epoch)
        tf.summary.scalar('phase3/e_reconstruction', avgs['e_recon'], step=epoch)
        tf.summary.scalar('phase3/d_real', avgs['d_real'], step=epoch)
        tf.summary.scalar('phase3/d_fake', avgs['d_fake'], step=epoch)
        # Derived metrics for balance monitoring
        tf.summary.scalar('phase3/d_total', avgs['d_real'] + avgs['d_fake'], step=epoch)
        tf.summary.scalar('phase3/g_total',
                          avgs['g_adv'] + LAMBDA_SUPERVISED * avgs['g_sup'] +
                          LAMBDA_RECONSTRUCTION * avgs['g_moment'] + avgs['e_recon'],
                          step=epoch)
        tf.summary.scalar('phase3/d_skip_pct', skip_pct, step=epoch)

    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d}/{JOINT_EPOCHS} | "
              f"G_adv: {avgs['g_adv']:.4f}  "
              f"G_sup: {avgs['g_sup']:.4f}  "
              f"G_mom: {avgs['g_moment']:.4f}  "
              f"E_rec: {avgs['e_recon']:.4f} | "
              f"D_real: {avgs['d_real']:.4f}  "
              f"D_fake: {avgs['d_fake']:.4f} | "
              f"D_skip: {skip_pct:.0f}%")

tb_writer.flush()
print("\nPhase 3 (Joint training) complete!")

Phase 3: Joint Adversarial Training (1000 epochs)
  D-skip threshold: 1.0
Epoch    0/1000 | G_adv: 0.7910  G_sup: 0.0150  G_mom: 0.0563  E_rec: 0.2689 | D_real: 0.7922  D_fake: 0.6435 | D_skip: 0%
Epoch  100/1000 | G_adv: 0.8063  G_sup: 0.0115  G_mom: 0.0064  E_rec: 0.3203 | D_real: 0.8178  D_fake: 0.6341 | D_skip: 0%


## 7. Training Loss Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# Phase 1
axes[0, 0].plot(embedding_history['reconstruction'], label='Reconstruction')
axes[0, 0].plot(embedding_history['supervised'], label='Supervised (×0.1)', alpha=0.7)
axes[0, 0].set_title('Phase 1: Embedding')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Phase 2
axes[0, 1].plot(supervised_history)
axes[0, 1].set_title('Phase 2: Supervised')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Supervisor Loss')
axes[0, 1].grid(True, alpha=0.3)

# Phase 3 — Adversarial
axes[0, 2].plot(joint_history['g_adv'], label='G adversarial', alpha=0.7)
axes[0, 2].plot(joint_history['d_real'], label='D real', alpha=0.7)
axes[0, 2].plot(joint_history['d_fake'], label='D fake', alpha=0.7)
axes[0, 2].set_title('Phase 3: Adversarial')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Phase 3 — Supervised
axes[1, 0].plot(joint_history['g_sup'])
axes[1, 0].set_title('Phase 3: Generator Supervised Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].grid(True, alpha=0.3)

# Phase 3 — Moment
axes[1, 1].plot(joint_history['g_moment'])
axes[1, 1].set_title('Phase 3: Moment Matching Loss')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].grid(True, alpha=0.3)

# Phase 3 — Reconstruction
axes[1, 2].plot(joint_history['e_recon'])
axes[1, 2].set_title('Phase 3: Reconstruction Loss')
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle('Hybrid WaveNet-TimeGAN Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Generate Synthetic Data

Generation pipeline: $Z \to \text{Generator} \to \hat{E} \to \text{Supervisor} \to \text{Recovery} \to \tilde{X}$

In [ ]:
n_samples = 500

# Generate: noise → generator → supervisor (temporal smoothing) → recovery (to data space)
Z = tf.random.normal([n_samples, SEQUENCE_LENGTH, LATENT_DIM])
E_hat = generator(Z, training=False)
H_hat = supervisor(E_hat, training=False)
synthetic_sequences = recovery(H_hat, training=False).numpy()

# Rescale to original log-return space
synthetic_rescaled = scaler.inverse_transform(
    synthetic_sequences.reshape(-1, 1)
).reshape(n_samples, SEQUENCE_LENGTH)

real_rescaled = scaler.inverse_transform(
    sequences[:5].reshape(-1, 1)
).reshape(5, SEQUENCE_LENGTH)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i in range(5):
    axes[0].plot(synthetic_rescaled[i], alpha=0.7, label=f'Synthetic {i+1}')
axes[0].set_title('Hybrid WaveNet-TimeGAN — Synthetic Sequences')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Log Return')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for i in range(5):
    axes[1].plot(real_rescaled[i], alpha=0.7, label=f'Real {i+1}')
axes[1].set_title('Real Sequences')
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Log Return')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Synthetic — mean: {synthetic_rescaled.mean():.6f}, std: {synthetic_rescaled.std():.6f}")
print(f"Real      — mean: {log_returns.mean():.6f}, std: {log_returns.std():.6f}")

## 9. Save Models and Data

In [ ]:
os.makedirs('../models', exist_ok=True)

for m in [embedder, recovery, supervisor, generator, discriminator]:
    m.save(f'../models/hybrid_{m.name.lower()}.keras')
    m.save_weights(f'../models/hybrid_{m.name.lower()}.weights.h5')

np.save('../data/processed/synthetic_hybrid_wavenet_timegan.npy', synthetic_sequences)
pd.DataFrame(
    synthetic_rescaled.flatten(), columns=['Synthetic_LogReturn']
).to_csv('../data/synthetic/hybrid_wavenet_timegan_synthetic_full.csv', index=False)

print("All models and synthetic data saved.")